In [ ]:
# pip install transformer_lens einops
from huggingface_hub import login
login(token="hf_YourTokenHere")  # Replace with your actual token
from transformer_lens import HookedTransformer
import torch
from einops import rearrange, repeat, einsum
import matplotlib.pyplot as plt
from math import sqrt

In [ ]:
device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cpu")
model = HookedTransformer.from_pretrained("google/gemma-2-2b", device=device, center_unembed=False, cache_dir="local_files/gemma-2-2b")
model = model.eval()

In [ ]:
def get_answer_token_indices(model, prefix, answer, sep=" "):
    """
    Returns:
        text: full prompt
        tokens: model tokens, shape (1, seq)
        answer_token_indices: token positions corresponding to `answer`
    """
    text = prefix + sep + answer

    # Use HF tokenizer offsets to map characters -> tokens
    enc = model.tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=False,
    )

    answer_start = len(prefix + sep)
    answer_end = len(text)

    raw_answer_token_indices = [
        i for i, (a, b) in enumerate(enc["offset_mapping"])
        if b > answer_start and a < answer_end
    ]

    tokens = model.to_tokens(text)

    # TransformerLens often prepends a BOS token.
    # This aligns tokenizer offsets with model.to_tokens output.
    n_extra = tokens.shape[-1] - len(enc["input_ids"])
    answer_token_indices = [i + n_extra for i in raw_answer_token_indices]

    return text, tokens, answer_token_indices
def get_residual_last_token(model, base_pattern, answers, pool="last"):
    """
    pool:
        "last" -> use residual after the last token of the month
        "mean" -> average residuals over all tokens of the month
    """
    all_residuals = {}

    for answer in answers:
        text, tokens, answer_token_indices = get_answer_token_indices(
            model,
            base_pattern,
            answer,
            sep=" ",
        )

        if len(answer_token_indices) == 0:
            raise ValueError(f"No token indices found for answer: {answer   !r}")

        with torch.no_grad():
            logits, cache = model.run_with_cache(tokens)

        residuals = []

        for layer in range(model.cfg.n_layers):
            resid = cache[f"blocks.{layer}.hook_resid_post"][0]  # [seq, d_model]
            
            if pool == "last":
                vec = resid[answer_token_indices[-1], :]
            elif pool == "mean":
                vec = resid[answer_token_indices, :].mean(dim=0)
            else:
                raise ValueError("pool must be 'last' or 'mean'")

            residuals.append(vec.cpu())

        all_residuals[answer] = torch.stack(residuals)

    return all_residuals

In [ ]:
months_eng = ["January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
base_pattern_eng = "The month of the year is"
months_eng_residuals = get_residual_last_token(model, base_pattern_eng, months_eng, pool="mean")

In [ ]:
months_fr = ["Janvier", "Février", "Mars", "Avril", "Mai", "Juin", "Juillet", "Août", "Septembre", "Octobre", "Novembre", "Décembre"]
base_pattern_fr = "Le mois de l'année est"
months_fr_residuals = get_residual_last_token(model, base_pattern_fr, months_fr, pool="mean")

In [ ]:
months_ita = ["Gennaio", "Febbraio", "Marzo", "Aprile", "Maggio", "Giugno", "Luglio", "Agosto", "Settembre", "Ottobre", "Novembre", "DDicembre"]
base_pattern_ita = "Il mese dell'anno è"
months_ita_residuals = get_residual_last_token(model, base_pattern_ita, months_ita, pool="mean")

In [ ]:
months_deu = ["Januar", "Februar", "März", "April", "Mai", "Juni", "Juli", "August", "September", "Oktober", "November", "Dezember"]
base_pattern_deu = "Der Monat des Jahres ist"
months_deu_residuals = get_residual_last_token(model, base_pattern_deu, months_deu, pool="mean")

In [ ]:
months_esp = ["Enero", "Febrero", "MMarzo", "Abril", "Mayo", "Junio", "Julio", "Agosto", "Septiembre", "Octubre", "Noviembre", "Diciembre"]
base_pattern_esp = "El mes del año es"
months_esp_residuals = get_residual_last_token(model, base_pattern_esp, months_esp, pool="mean")

In [ ]:
months_rus = ["январь", "февраль", "март", "апрель", "май", "июнь", "июль", "август", "сентябрь", "октябрь", "ноябрь", "декабрь"]
base_pattern_rus = "месяц года"
months_rus_residuals = get_residual_last_token(model, base_pattern_rus, months_rus, pool="mean")

In [ ]:
months_eng_data = torch.stack([val for key, val in months_eng_residuals.items()])
months_deu_data = torch.stack([val for key, val in months_deu_residuals.items()])
months_esp_data = torch.stack([val for key, val in months_esp_residuals.items()])
months_fr_data = torch.stack([val for key, val in months_fr_residuals.items()])
months_ita_data = torch.stack([val for key, val in months_ita_residuals.items()])
months_rus_data = torch.stack([val for key, val in months_rus_residuals.items()])   
months_data = torch.cat([
    months_eng_data[None, :, :, :],
    months_deu_data[None, :, :, :],
    months_esp_data[None, :, :, :],
    months_fr_data[None, :, :, :],
    months_ita_data[None, :, :, :],
    months_rus_data[None, :, :, :],
], dim=0)

In [ ]:
from scipy.linalg import orthogonal_procrustes
reference_idx = 2
proj_matr = torch.eye(months_data.shape[1], device=months_data.device) - (1/months_data.shape[1]) * torch.ones((months_data.shape[1], months_data.shape[1]), device=months_data.device)
labels = ["English", "German", "Italian", "French", "Spanish", "Russian"]
precision = []
fano = []
correlations = []
mutual = []
fig, axs = plt.subplots(3, model.cfg.n_layers, figsize=(40, 6))
for layer in range(model.cfg.n_layers):
    layer_data = months_data[:, :, layer, :]
    all_grams = layer_data.view(-1, layer_data.size(-1)) @ layer_data.view(-1, layer_data.size(-1)).T
    all_grams = all_grams.view(layer_data.size(0), 12, layer_data.size(0), 12).permute(0, 2, 1, 3).contiguous()
    self_grams = all_grams.diagonal(dim1=0, dim2=1).permute(2,0, 1).contiguous()
    self_grams = self_grams / self_grams.diagonal(dim1=-2, dim2=-1).sqrt()[:, :, None] / self_grams.diagonal(dim1=-2, dim2=-1).sqrt()[:, None, :]
    self_grams_proj = torch.einsum("ia,sab,bj->sij", proj_matr, self_grams, proj_matr)
    vals, vecs = torch.linalg.eigh(self_grams_proj)
    coords_2d = vecs[:, :, -2:]
    angle_matrix = torch.zeros((months_data.size(0), months_data.size(0)), device=months_data.device)
    rotation_matrices = torch.zeros((months_data.size(0), months_data.size(0), 2, 2), device=months_data.device)
    for i in range(coords_2d.size(0)):
        if i == reference_idx:
            axs[2, layer].scatter(coords_2d[i,:,0].cpu(), coords_2d[i,:,1].cpu(), color='black')
        for j in range(coords_2d.size(0)):
            if i < j:
                rot_ij, scale_ij = orthogonal_procrustes(coords_2d[j].cpu(), coords_2d[i].cpu())
                rot_ij = torch.from_numpy(rot_ij).to(coords_2d.device)
                angle_ij = torch.atan2(rot_ij[0,1], rot_ij[0,0])
                angle_matrix[i,j] = angle_ij
                angle_matrix[j,i] = -angle_ij
                rotation_matrices[i,j] = rot_ij
                rotation_matrices[j,i] = rot_ij.T
                if i == reference_idx and j  == reference_idx + 1:
                    coords_j_rot = (coords_2d[j] - coords_2d[j].mean(dim=0, keepdim=True)) @ rot_ij.T + coords_2d[j].mean(dim=0, keepdim=True)
                    axs[2, layer].plot(coords_j_rot[:,0].cpu(), coords_j_rot[:,1].cpu(), lw=1)

    axs[1,layer].imshow(torch.sin(angle_matrix), vmin=-.5, vmax=.5, cmap='twilight_shifted')
    
    mean_all_grams = all_grams.mean(dim=(-2, -1))
    mean_all_grams_diag = mean_all_grams.diag()
    mean_all_grams = mean_all_grams/mean_all_grams_diag.sqrt()[:, None] / mean_all_grams_diag.sqrt()[None, :]
    axs[0,layer].imshow(mean_all_grams, vmin=0.0, vmax=1, cmap='coolwarm')
    if layer == 0:
        axs[0,layer].set_xticks(ticks=range(len(labels)), labels=labels, rotation=45, fontsize=8)
        axs[0,layer].set_yticks(ticks=range(len(labels)), labels=labels, fontsize=8)
    else:
        axs[0,layer].set_xticks([])
        axs[0,layer].set_yticks([])
    indices_to_use = torch.argwhere(torch.arange(mean_all_grams.size(0)) != reference_idx).squeeze()
    matrix_indices_to_use = torch.meshgrid(indices_to_use, indices_to_use, indexing="ij")
    avg_correlation = mean_all_grams[reference_idx,1:].mean()
    var_mut = (1/mean_all_grams[reference_idx,reference_idx]) * mean_all_grams[reference_idx,indices_to_use]  @ torch.linalg.inv(mean_all_grams[matrix_indices_to_use]) @ mean_all_grams[indices_to_use,reference_idx]
    mut = -0.5*torch.log(1 - var_mut)
    precision.append((mut**2)/var_mut)
    fano.append(mut/var_mut)
    correlations.append(avg_correlation)
    mutual.append(mut)
precision = torch.stack(precision)
fano = torch.stack(fano)
correlations = torch.stack(correlations)
mutual = torch.stack(mutual)
plt.show()

In [ ]:
plt.plot(torch.arange(0, len(fano)), fano/2)
plt.plot(torch.arange(0, len(correlations)), (1+correlations)/2)
plt.xlabel("Layer")
plt.ylabel("English vs Non-English")
plt.ylim(0, 1.0)
plt.show()

In [ ]:
layer_to_use = 6
months_eng_gram = torch.cov(months_eng_data[:,layer_to_use,:])
months_eng_gram_diag = torch.diag(months_eng_gram)
months_eng_gram = months_eng_gram/ torch.sqrt(months_eng_gram_diag[:, None] * months_eng_gram_diag[None, :])
months_fr_gram = torch.cov(months_fr_data[:,layer_to_use,:])
months_fr_gram_diag = torch.diag(months_fr_gram)
months_fr_gram = months_fr_gram/ torch.sqrt(months_fr_gram_diag[:, None] * months_fr_gram_diag[None, :])
months_ita_gram = torch.cov(months_ita_data[:,layer_to_use,:])
months_ita_gram_diag = torch.diag(months_ita_gram)
months_ita_gram = months_ita_gram/ torch.sqrt(months_ita_gram_diag[:, None] * months_ita_gram_diag[None, :])
months_deu_gram = torch.cov(months_deu_data[:,layer_to_use,:])
months_deu_gram_diag = torch.diag(months_deu_gram)
months_deu_gram = months_deu_gram/ torch.sqrt(months_deu_gram_diag[:, None] * months_deu_gram_diag[None, :])
months_esp_gram = torch.cov(months_esp_data[:,layer_to_use,:])
months_esp_gram_diag = torch.diag(months_esp_gram)
months_esp_gram = months_esp_gram/ torch.sqrt(months_esp_gram_diag[:, None] * months_esp_gram_diag[None, :])
months_rus_gram = torch.cov(months_rus_data[:,layer_to_use,:])
months_rus_gram_diag = torch.diag(months_rus_gram)
months_rus_gram = months_rus_gram/ torch.sqrt(months_rus_gram_diag[:, None] * months_rus_gram_diag[None, :])

In [ ]:
wrapped_months =  torch.cat([torch.arange(12), torch.zeros(1, dtype=torch.long)], dim=0)
fig, axs = plt.subplots(1, 6, figsize=(20, 4))
proj_matr = torch.eye(months_eng_gram.shape[0]) - torch.ones(months_eng_gram.shape[0], months_eng_gram.shape[0])/months_eng_gram.shape[0]
proj_gram = proj_matr @ months_eng_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[0].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="o", label="English", cmap="coolwarm", s=40, edgecolor="black")
axs[0].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[0].set_title("English")

proj_matr = torch.eye(months_fr_gram.shape[0]) - torch.ones(months_fr_gram.shape[0], months_fr_gram.shape[0])/months_fr_gram.shape[0]
proj_gram = proj_matr @ months_fr_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[1].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="o", label="French", cmap="coolwarm", s=40, edgecolor="black")
axs[1].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[1].set_title("French")

proj_matr = torch.eye(months_ita_gram.shape[0]) - torch.ones(months_ita_gram.shape[0], months_ita_gram.shape[0])/months_ita_gram.shape[0]
proj_gram = proj_matr @ months_ita_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[2].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="^", label="Italian", cmap="coolwarm", s=40, edgecolor="black")
axs[2].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[2].set_title("Italian")

proj_matr = torch.eye(months_deu_gram.shape[0]) - torch.ones(months_deu_gram.shape[0], months_deu_gram.shape[0])/months_deu_gram.shape[0]
proj_gram = proj_matr @ months_deu_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[3].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="*", label="German", cmap="coolwarm", s=40, edgecolor="black")
axs[3].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[3].set_title("German")

proj_matr = torch.eye(months_esp_gram.shape[0]) - torch.ones(months_esp_gram.shape[0], months_esp_gram.shape[0])/months_esp_gram.shape[0]
proj_gram = proj_matr @ months_esp_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[4].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="D", label="Spanish", cmap="coolwarm", s=40, edgecolor="black")
axs[4].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[4].set_title("Spanish")

proj_matr = torch.eye(months_rus_gram.shape[0]) - torch.ones(months_rus_gram.shape[0], months_rus_gram.shape[0])/months_rus_gram.shape[0]
proj_gram = proj_matr @ months_rus_gram @ proj_matr
vals, vecs = torch.linalg.eigh(proj_gram)
axs[5].scatter(vecs[:,-1],vecs[:,-2], c=torch.arange(vecs.shape[0]), marker="s", label="Russian", cmap="coolwarm", s=40, edgecolor="black")
axs[5].plot(vecs[wrapped_months,-1],vecs[wrapped_months,-2], c="gray", alpha=0.5, zorder=-1)
axs[5].set_title("Russian") 
plt.show()